# Soft/untruncated many-body Lyapunov campaign

Nx=20; Ny=20,24,30,36,44,56,60; S=100 per size; T=4Ny. The runner uses 38-GiB-capped resident A100 batches, five-trajectory durable result shards, a rolling ten-cycle state checkpoint, and live outer/inner `tqdm` progress. Existing compatible datasets are not overwritten or pooled.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import torch

CONSTRUCTION = 'soft'
LANE_NAME = None
NY_LANE = None
REPORT_ONLY = False
MAX_NEW_EXECUTION_BATCHES = None  # Set to 1 for a worst-size timing test.
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/13_maxmix_manybody_lyapunov_4ny')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs/maxmix_manybody_lyapunov_nx20_ny20-60_hard-soft_s100_4ny_gpu_v4_38gib_memory_scaled')
LOCAL_BUNDLE_DIR = Path('/content/13_maxmix_manybody_lyapunov_4ny_soft')
SCRATCH_ROOT = Path('/content/maxmix_manybody_lyapunov_4ny_scratch_soft')
CONFIG = {'Nx': 20, 'Ny_values': [20, 24, 30, 36, 44, 56, 60], 'alpha_1': 1.0, 'alpha_2': 30.0, 'backend': 'local', 'constructions': {'hard': {'DW': True, 'dw_truncation': True, 'meas_slab_only': True}, 'soft': {'DW': True, 'dw_truncation': False, 'meas_slab_only': False}}, 'cycles_multiplier': 4, 'device': 'cuda:0', 'dtype': 'complex128', 'execution_batch_size_by_Ny': {'20': 100, '24': 100, '30': 100, '36': 90, '44': 60, '56': 40, '60': 35}, 'filling_fraction': 0.5, 'gpu_memory_hard_limit_gib': 38.0, 'init_mode': 'maxmix', 'n_a': 0.5, 'nshell': 1, 'observables': {'cap_tolerance': 1e-09, 'cumulative_log_probability': True, 'leading_log_sigma2_levels': 64, 'measurement_log_probability': True, 'occupation_spectrum': True, 'record_cycles': '0..4Ny inclusive', 'save_covariance_history': False, 'save_final_covariance': False, 'soft_mode_count': 16, 'spectrum_cycles': 'stride 4 union Ny,2Ny,3Ny,4Ny', 'squared_singular_value_convention': 'ell_i=log(sigma_i^2)', 'transverse_mode_profiles': True}, 'observer_sample_chunk_by_Ny': {'20': 20, '24': 20, '30': 10, '36': 10, '44': 5, '56': 2, '60': 2}, 'perfect_correction': True, 'postselect': False, 'result_shard_size': 5, 'resume_initial_purity_tolerance': 0.50000001, 'root_seed': 2026091001, 'samples_per_case': 100, 'sampling_revision': 'maxmix_manybody_lyapunov_nx20_ny20-60_hard-soft_s100_4ny_gpu_v4_38gib_memory_scaled', 'segment_cycles': 10, 'sequence': 'raster_y', 'state_representation': 'covariance', 'trial_orbitals': 'X'}
if not torch.cuda.is_available():
    raise RuntimeError('This production notebook requires CUDA.')
gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if 'A100' not in gpu_name or gpu_gib < 35:
    raise RuntimeError(f'Expected an A100 40-GB-class GPU, got {gpu_name} ({gpu_gib:.2f} GiB)')
if torch.complex128 != getattr(torch, CONFIG['dtype']):
    raise RuntimeError('Locked dtype is not complex128.')
print(json.dumps({
    'construction': CONSTRUCTION, 'lane': LANE_NAME, 'Ny_lane': NY_LANE,
    'config': CONFIG,
    'bundle': str(BUNDLE_DRIVE_DIR), 'output': str(OUTPUT_ROOT),
    'scratch': str(SCRATCH_ROOT), 'gpu': gpu_name, 'gpu_GiB': gpu_gib,
    'gpu_memory_hard_limit_GiB': CONFIG['gpu_memory_hard_limit_gib'],
    'execution_batch_size_by_Ny': CONFIG['execution_batch_size_by_Ny'],
    'report_only': REPORT_ONLY,
    'max_new_execution_batches': MAX_NEW_EXECUTION_BATCHES,
}, indent=2, sort_keys=True))


In [ ]:
import importlib.util
import shutil
import sys

if not BUNDLE_DRIVE_DIR.is_dir():
    raise FileNotFoundError(f'Missing Drive bundle: {BUNDLE_DRIVE_DIR}')
if LOCAL_BUNDLE_DIR.exists():
    shutil.rmtree(LOCAL_BUNDLE_DIR)
shutil.copytree(BUNDLE_DRIVE_DIR, LOCAL_BUNDLE_DIR)
config_path = Path(f'/content/maxmix_manybody_lyapunov_4ny_{CONSTRUCTION}.json')
config_path.write_text(json.dumps(CONFIG, indent=2, sort_keys=True) + '\n', encoding='utf-8')
runner_args = [
    '--config', str(config_path), '--construction', CONSTRUCTION,
    '--output-root', str(OUTPUT_ROOT), '--scratch-root', str(SCRATCH_ROOT),
]
if REPORT_ONLY:
    runner_args.append('--report-only')
if MAX_NEW_EXECUTION_BATCHES is not None:
    runner_args.extend(['--max-new-execution-batches', str(int(MAX_NEW_EXECUTION_BATCHES))])
runner_path = LOCAL_BUNDLE_DIR / 'run_campaign.py'
print('[launch in notebook kernel] ' + str(runner_path) + ' ' + ' '.join(runner_args), flush=True)
module_name = f'_maxmix_manybody_lyapunov_runner_{CONSTRUCTION}'
sys.modules.pop(module_name, None)
sys.path.insert(0, str(LOCAL_BUNDLE_DIR))
spec = importlib.util.spec_from_file_location(module_name, runner_path)
if spec is None or spec.loader is None:
    raise RuntimeError(f'Could not load staged runner: {runner_path}')
runner = importlib.util.module_from_spec(spec)
sys.modules[module_name] = runner
spec.loader.exec_module(runner)
if NY_LANE is not None:
    _unfiltered_expand = runner.expand_execution_batches
    _lane_set = frozenset(int(value) for value in NY_LANE)
    def _lane_expand(config, construction):
        expanded = _unfiltered_expand(config, construction)
        selected = [task for task in expanded if task.ny in _lane_set]
        if {task.ny for task in selected} != _lane_set:
            raise RuntimeError(f'Lane {LANE_NAME} did not resolve every requested Ny: {NY_LANE}')
        return selected
    runner.expand_execution_batches = _lane_expand
returncode = runner.main(runner_args)
if returncode:
    raise RuntimeError(f'Runner exited with status {returncode}')
print(f'[notebook] {CONSTRUCTION}-wall runner exited successfully', flush=True)


In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
